In [68]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt

if torch.cuda.is_available():
    device="cuda"
else:
    device="cpu"
device

'cpu'

In [69]:
data = pd.read_csv('./structuralparameters-vs-capacities-4500mofs.csv', index_col=False, header=0)
data

,usablegc,usablevc,density,porosity,Largestporeradius,SSA,SPV,du,metalw,metala
0,0.2294,0.003505,1.525,0.04,2.22,632,0.026,289.0,19.449,3.704
1,0.1965,0.003082,1.566,0.02,1.77,529,0.013,289.0,19.449,3.704
2,1.3542,0.013027,0.949,0.30,5.70,3024,0.316,217.0,24.053,7.143
3,0.3773,0.004750,1.255,0.04,2.27,1002,0.032,233.0,15.330,2.439
4,0.4118,0.005179,1.253,0.04,2.41,1085,0.032,465.0,15.330,2.439
...,...,...,...,...,...,...,...,...,...,...
4495,0.1575,0.002740,1.737,0.10,2.20,572,0.058,473.0,29.160,2.469
4496,0.1573,0.002793,1.773,0.10,2.18,561,0.056,473.0,29.932,2.469
4497,0.2423,0.003802,1.565,0.04,2.24,539,0.026,321.0,11.644,3.333
4498,0.5283,0.005848,1.102,0.04,2.56,1639,0.036,217.0,16.467,1.408


In [70]:
limites_tabla = pd.read_csv('./limits.csv', index_col=False, header=0)
limites_tabla

,Variable,Units,lowlimit,highlimit
0,usablegc,(wt.%),0.000001,15.00
1,usablevc,(kg/L),0.000001,0.06
2,density,(kg/L),0.100000,5.00
3,porosity,(dimensionless),0.001000,0.90
4,R_i,(angstroms),1.000000,40.00
5,ssa,(m^2/g),100.000000,10000.00
6,specificporevolume,(cm^3/g),0.001000,10.00
7,du,(dimensionless),-1000.000000,4000.00
8,metalw,%,0.000001,90.00
9,metala,%,0.000001,90.00


In [71]:
limit_inputs = np.array(limites_tabla.iloc[0:2, -2:]).T
limit_inputs

array([[1.0e-06, 1.0e-06],
       [1.5e+01, 6.0e-02]])

In [72]:
limit_outputs = np.array(limites_tabla.iloc[2:, -2:]).T
limit_outputs

array([[ 1.e-01,  1.e-03,  1.e+00,  1.e+02,  1.e-03, -1.e+03,  1.e-06,
         1.e-06],
       [ 5.e+00,  9.e-01,  4.e+01,  1.e+04,  1.e+01,  4.e+03,  9.e+01,
         9.e+01]])

In [73]:
escaler_X = MinMaxScaler().fit(limit_inputs)
escaler_y = MinMaxScaler().fit(limit_outputs)

In [74]:
X = torch.tensor(escaler_X.transform(np.array(data.iloc[:, 0:2])), dtype=torch.float32)
X.max(dim=0), X.min(dim=0), X.shape, type(X), X

(torch.return_types.max(
 values=tensor([0.7427, 0.3981]),
 indices=tensor([245,  82])),
 torch.return_types.min(
 values=tensor([0.0010, 0.0040]),
 indices=tensor([2600, 2600])),
 torch.Size([4500, 2]),
 torch.Tensor,
 tensor([[0.0153, 0.0584],
         [0.0131, 0.0514],
         [0.0903, 0.2171],
         ...,
         [0.0162, 0.0634],
         [0.0352, 0.0975],
         [0.0360, 0.0983]]))

In [75]:
y = torch.tensor(escaler_y.fit_transform(np.array(data.iloc[:, 2:])), dtype=torch.float32)
y.max(dim=0), y.min(dim=0), y.shape, type(y),y, y.shape

(torch.return_types.max(
 values=tensor([1., 1., 1., 1., 1., 1., 1., 1.]),
 indices=tensor([3294, 2996, 2996,  245,  245,   61, 4443, 1919])),
 torch.return_types.min(
 values=tensor([0., 0., 0., 0., 0., 0., 0., 0.]),
 indices=tensor([ 245,    1, 3189, 1995, 3294, 1006, 3894,  329])),
 torch.Size([4500, 8]),
 torch.Tensor,
 tensor([[0.3612, 0.0233, 0.0215,  ..., 0.1365, 0.2564, 0.0956],
         [0.3719, 0.0000, 0.0084,  ..., 0.1365, 0.2564, 0.0956],
         [0.2120, 0.3256, 0.1229,  ..., 0.1141, 0.3233, 0.2006],
         ...,
         [0.3716, 0.0233, 0.0221,  ..., 0.1464, 0.1429, 0.0843],
         [0.2516, 0.0233, 0.0315,  ..., 0.1141, 0.2130, 0.0255],
         [0.2477, 0.0930, 0.0326,  ..., 0.1141, 0.2130, 0.0255]]),
 torch.Size([4500, 8]))

In [76]:
idx_train, idx_test = train_test_split(np.arange(4500), test_size=1/3, random_state=42)

In [77]:
class SVR_regressor:
    def __init__(self, n_outputs=5):
        # Constructor
        self.n_outputs = n_outputs
        self.regressor = [SVR() for i in range(n_outputs)]
    
    def fit(self, X, y):
        for i in range(self.n_outputs):
            self.regressor[i].fit(X, y[:, i])

    def predict(self, X):
        y = np.zeros((X.shape[0], self.n_outputs), dtype=float)
        for i in range(self.n_outputs):
            y[:, i] = self.regressor[i].predict(X)
        return y

In [83]:
regressor_svm = SVR_regressor(n_outputs=8)

In [84]:
regressor_svm.fit(X[idx_train], y[idx_train])

In [85]:
y_train_predict = regressor_svm.predict(X[idx_train])

In [86]:
y_train_predict.shape

(3000, 8)

In [87]:
r2_score(y_pred=y_train_predict, y_true=y[idx_train])

0.04855292124518035

In [88]:
y_test_predict = regressor_svm.predict(X[idx_test])

In [89]:
r2_score(y_pred=y_test_predict, y_true=y[idx_test])

0.1328886793155239

In [90]:
class TREE_regressor:
    def __init__(self, n_outputs=5):
        # Constructor
        self.n_outputs = n_outputs
        self.regressor = [DecisionTreeRegressor() for i in range(n_outputs)]
    
    def fit(self, X, y):
        for i in range(self.n_outputs):
            self.regressor[i].fit(X, y[:, i])

    def predict(self, X):
        y = np.zeros((X.shape[0], self.n_outputs), dtype=float)
        for i in range(self.n_outputs):
            y[:, i] = self.regressor[i].predict(X)
        return y

In [92]:
regressor_tree = TREE_regressor(n_outputs=8)

In [93]:
regressor_tree.fit(X[idx_train], y[idx_train])

In [94]:
y_tree_predict_train = regressor_tree.predict(X[idx_train])

In [95]:
r2_score(y_pred=y_tree_predict_train, y_true=y[idx_train])

1.0

In [96]:
y_tree_predict_test = regressor_tree.predict(X[idx_test])

In [97]:
r2_score(y_pred=y_tree_predict_test, y_true=y[idx_test])

0.4145745138524278